In [12]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
df = pd.read_excel(r"C:\Users\ODAMA\Downloads\01_Shoprite_Nigeria_Sales_Dashboard.xlsx",
                   sheet_name='Sales Data',
                   header=2)
df['Month'] = df['Month'].astype(str)

# ── KPIs ──────────────────────────────────────────────────────────────────────
total_revenue   = df['Total Sales (₦)'].sum()
total_cogs      = df['COGS (₦)'].sum()
total_profit    = df['Profit (₦)'].sum()
total_customers = df['Customers'].sum()

# ── CHART DATA ────────────────────────────────────────────────────────────────
product_profit = df.groupby('Product')['Profit (₦)'].sum().reset_index().sort_values('Profit (₦)')

region_sales = df.groupby('Store Location')['Total Sales (₦)'].sum().reset_index().nlargest(4, 'Total Sales (₦)')

sales_rep = df.groupby('Sales Rep')['Total Sales (₦)'].sum().reset_index().nlargest(6, 'Total Sales (₦)')

month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_customers = df.groupby('Month')['Customers'].sum().reset_index()
monthly_customers['Month'] = pd.Categorical(monthly_customers['Month'], categories=month_order, ordered=True)
monthly_customers = monthly_customers.sort_values('Month')

# ── CHARTS ────────────────────────────────────────────────────────────────────
fig_product = px.bar(product_profit, x='Profit (₦)', y='Product', orientation='h',
                     color='Profit (₦)', color_continuous_scale='Blues',
                     title='PRODUCT BY PROFIT')

fig_city = px.pie(region_sales, names='Store Location', values='Total Sales (₦)', hole=0.5,
                  color_discrete_sequence=['#08306b','#2171b5','#6baed6','#c6dbef'],
                  title='STORE LOCATION BY SALES')

fig_sales_rep = px.bar(sales_rep, x='Sales Rep', y='Total Sales (₦)',
                       color='Total Sales (₦)', color_continuous_scale='Blues',
                       title='SALES REP BY REVENUE')

fig_month = px.line(monthly_customers, x='Month', y='Customers',
                    markers=True, title='CUSTOMERS BY MONTH')
fig_month.update_traces(line_color='#1f6fbf', marker=dict(color='#1f6fbf', size=8))

# ── STYLE ALL CHARTS ──────────────────────────────────────────────────────────
for fig in [fig_product, fig_city, fig_sales_rep, fig_month]:
    fig.update_layout(
        plot_bgcolor='#eaf3fb',
        paper_bgcolor='#ffffff',
        font=dict(color='#1f3b5c', family='Arial'),
        title_font=dict(size=12, color='#1f3b5c'),
        margin=dict(l=10, r=10, t=35, b=10),
        height=250,
        showlegend=True
    )

# ── STYLES
BLUE_DARK  = '#1f3b5c'
BLUE_MID   = '#2171b5'
BLUE_LIGHT = '#d6e6f2'
WHITE      = '#ffffff'

kpi_card = {
    'backgroundColor': WHITE, 'padding': '8px 14px', 'borderRadius': '8px',
    'textAlign': 'center', 'flex': '1', 'margin': '4px',
    'border': f'1px solid {BLUE_LIGHT}', 'boxShadow': '2px 2px 5px rgba(0,0,0,0.08)'
}
chart_box = {
    'backgroundColor': WHITE, 'padding': '6px', 'borderRadius': '8px',
    'boxShadow': '2px 2px 5px rgba(0,0,0,0.08)', 'flex': '1'
}
sidebar_label = {
    'backgroundColor': BLUE_MID, 'color': WHITE, 'padding': '4px 8px',
    'borderRadius': '4px', 'marginBottom': '5px', 'fontSize': '12px'
}

# ── APP LAYOUT ────────────────────────────────────────────────────────────────
app = Dash(__name__)

app.layout = html.Div(
    style={'backgroundColor': '#eaf3f9', 'padding': '10px', 'fontFamily': 'Arial'},
    children=[

        # Title + KPI Row
        html.Div([
            html.Div(
                html.H3('SHOPRITE NIGERIA — SALES INSIGHT DASHBOARD',
                        style={'color': WHITE, 'margin': '0', 'fontSize': '15px'}),
                style={'backgroundColor': BLUE_DARK, 'padding': '10px 16px',
                       'borderRadius': '8px', 'flex': '2', 'marginRight': '8px'}
            ),
            html.Div([
                html.Div([html.P('REVENUE',   style={'margin':'0','fontSize':'10px','color':BLUE_MID}),
                          html.H5(f'₦{total_revenue:,.0f}', style={'margin':'0','color':BLUE_DARK,'fontSize':'12px'})], style=kpi_card),
                html.Div([html.P('COGS',      style={'margin':'0','fontSize':'10px','color':BLUE_MID}),
                          html.H5(f'₦{total_cogs:,.0f}',   style={'margin':'0','color':BLUE_DARK,'fontSize':'12px'})], style=kpi_card),
                html.Div([html.P('PROFIT',    style={'margin':'0','fontSize':'10px','color':BLUE_MID}),
                          html.H5(f'₦{total_profit:,.0f}', style={'margin':'0','color':BLUE_DARK,'fontSize':'12px'})], style=kpi_card),
                html.Div([html.P('CUSTOMERS', style={'margin':'0','fontSize':'10px','color':BLUE_MID}),
                          html.H5(f'{total_customers:,.0f}',style={'margin':'0','color':BLUE_DARK,'fontSize':'12px'})], style=kpi_card),
            ], style={'display': 'flex', 'flex': '4'})
        ], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '8px'}),

        # Body: Sidebar + Charts
        html.Div([

            # Sidebar
            html.Div([
                html.P('Location', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'4px','fontSize':'11px'}),
                *[html.Div(r, style=sidebar_label) for r in ['Lagos','Abuja','Kano','Port Harcourt']],
                html.Br(),
                html.P('Category', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'4px','fontSize':'11px'}),
                *[html.Div(c, style=sidebar_label) for c in ['Groceries','Electronics','Clothing','Home & Kitchen']],
                html.Br(), html.Br(),
                html.P('Prepared by', style={'fontSize':'10px','color':BLUE_DARK,'margin':'0'}),
                html.P('Odama Joseph', style={'fontSize':'10px','color':BLUE_DARK,'fontWeight':'bold','margin':'0'}),
            ], style={
                'width': '140px', 'backgroundColor': WHITE, 'padding': '10px',
                'borderRadius': '8px', 'boxShadow': '2px 2px 5px rgba(0,0,0,0.08)',
                'marginRight': '8px'
            }),

            # Charts Grid
            html.Div([
                html.Div([
                    html.Div(dcc.Graph(figure=fig_product,   config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_city,      config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '8px'}),

                html.Div([
                    html.Div(dcc.Graph(figure=fig_sales_rep, config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_month,     config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '8px', 'marginTop': '8px'}),
            ], style={'flex': '1'})

        ], style={'display': 'flex', 'alignItems': 'flex-start'})
    ]
)

# ── RUN ───────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    app.run(debug=True, port=8052)